In [1]:
# MPAGE Package Testing
# This notebook demonstrates basic functionality of the MPAGE package

# Load the package
devtools::load_all()


ℹ Loading MPAGE




Warning message:
“Objects listed as exports, but not present in namespace:
• add_custom_proteins
• annotate_modules
• filter_ppi_network
• filter_rna_modules
• get_rna_mod_references
• identify_modules
• list_modification_types
• list_organisms
• load_modules
• load_ppi_network
• save_modules
• save_ppi_network
• save_rna_mod_proteins
• validate_protein_file
• preprocess_intact_data
• load_intact_network
• download_intact_data”


In [2]:
library(clusterProfiler)
library(org.Hs.eg.db)


clusterProfiler v4.16.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

Guangchuang Yu, Li-Gen Wang, Yanyan Han and Qing-Yu He.
clusterProfiler: an R package for comparing biological themes among
gene clusters. OMICS: A Journal of Integrative Biology. 2012,
16(5):284-287


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter


Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    coln

In [3]:
rna_mod_proteins <- get_rna_mod_proteins()


In [4]:
networkds <- build_ppi_network()
merged_networks <- merge_ppi_networks(networkds, merge_method = "union", add_source_labels = TRUE)


Processing STRING database...



STRING Network Summary:


Processed 6857702 unique interactions

Unique proteins: 19622



Nodes: 16185 
Edges: 236000 
Average degree: 29.16281 


Processing BIOGRID database...



...

INTACT Network Summary:


Processed 606332 unique interactions

Unique proteins: 18416



Nodes: 5597 
Edges: 20870 
Average degree: 7.457567 


Merging 3 networks using method: union



=== Merged Network Summary ===
Data sources: string, biogrid, intact
Merge method: union
Total nodes: 21604
Total edges: 1079304
Network density: 0.0046
Average degree: 99.92

=== Source Contribution ===

biogrid  intact  string 
 904639   11187  236000 

Interactions from multiple sources: 66409 (6.2%)


In [5]:
sub_modules <- identify_modules_iterative(merged_networks,
    min_module_size = 1,
    max_module_size = 50,
    target_proteins = rna_mod_proteins$gene_symbol
)


Identified 40 modules containing target proteins

Total target proteins covered: 54



In [ ]:
saveRDS(sub_modules, file = "inst/extdata/rmp_modules.rds")


In [6]:
for (i in seq_along(sub_modules)) {
    # print(intersect(sub_modules[[i]]$genes, rna_mod_proteins$gene_symbol))
    # print(sub_modules[[i]]$genes)
    print(rna_mod_proteins %>% dplyr::filter(gene_symbol %in% sub_modules[[i]]$genes))
}


  gene_symbol modification_type functional_role
1       NSUN3               m5C          writer
  gene_symbol modification_type functional_role
1       RPS25               Psi          reader
2       RPS26               Psi          reader
  gene_symbol modification_type functional_role
1      YTHDC2               m6A          reader
  gene_symbol modification_type functional_role
1      ADARB1            A-to-I          writer
  gene_symbol modification_type functional_role
1        DKC1               Psi          writer
  gene_symbol modification_type functional_role
1       NSUN5               m5C          writer
2       NSUN6               m5C          writer
3       NSUN7               m5C          writer
  gene_symbol modification_type functional_role
1       TRMT6               m1A          writer
  gene_symbol modification_type functional_role
1        YBX1               m5C          reader
  gene_symbol modification_type functional_role
1     IGF2BP1               m6A         

In [7]:
sub_modules[[1]]$genes


[1] "ABCB6"      "ARL5B"      "ATP12A"     "ATP1B1"     "ATP7B"     
 [6] "ATP8B2"     "C1ORF74"    "CD164L2"    "CD226"      "CD3D"      
[11] "CLEC2D"     "CNNM4"      "DGCR2"      "DNAJC11"    "DSTYK"     
[16] "FAM63B"     "GRPR"       "HSDL1"      "ITM2A"      "JTB"       
[21] "KIAA1586"   "KLRC4"      "KLRD1"      "KLRF1"      "MBLAC2"    
[26] "MICB"       "MTCH1"      "MTCH2"      "NCR2"       "NKAIN2"    
[31] "NSUN3"      "PLD6"       "PPTC7"      "RHOBTB3"    "RHOT1"     
[36] "SAYSD1"     "SCCPDH"     "SLC25A16"   "SLC25A20"   "SLC25A33"  
[41] "SLC35D2"    "TACR3"      "TMEM184B"   "TRABD"      "TYROBP"    
[46] "USP30"      "SLC25A10-2"

In [8]:
print_enrichment_summary(functional_enrichment(gene_set = sub_modules[[1]]$genes, key_proteins = sub_modules[[1]]$target_proteins))


'select()' returned 1:1 mapping between keys and columns

'select()' returned 1:many mapping between keys and columns

Running functional enrichment analysis...

Running KEGG enrichment...

Reading KEGG annotation online: "https://rest.kegg.jp/link/hsa/pathway"...

Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/hsa"...

Running GO-Biological Process enrichment...

Running WikiPathways enrichment...

Enrichment analysis completed!



=== Functional Enrichment Analysis Summary ===
Input genes: 47
Mapped genes: 44
Organism: hsa

=== KEGG Enrichment (1 significant pathways) ===
1. Natural killer cell mediated cytotoxicity (p=8.00e-05, q=3.60e-03, genes=4)

=== GO_BP Enrichment (41 significant pathways) ===
1. mitochondrion organization (p=4.79e-07, q=3.58e-04, genes=9)
2. mitochondrial fusion (p=1.07e-06, q=4.00e-04, genes=4)
3. mitochondrial transport (p=3.07e-06, q=7.65e-04, genes=6)
4. positive regulation of leukocyte mediated cytotoxicity (p=1.58e-05, q=1.75e-03, genes=4)
5. stimulatory C-type lectin receptor signaling pathway (p=1.78e-05, q=1.75e-03, genes=3)
6. response to lectin (p=1.78e-05, q=1.75e-03, genes=3)
7. cellular response to lectin (p=1.78e-05, q=1.75e-03, genes=3)
8. positive regulation of cell killing (p=1.87e-05, q=1.75e-03, genes=4)
9. positive regulation of natural killer cell mediated cytotoxicity (p=3.64e-05, q=3.02e-03, genes=3)
10. positive regulation of natural killer cell mediated immunity

In [9]:
ids <- bitr(sub_modules[[1]]$genes, fromType = "SYMBOL", toType = c("ENTREZID", "ENSEMBL"), OrgDb = "org.Hs.eg.db")
ids


'select()' returned 1:many mapping between keys and columns

Warning message in bitr(sub_modules[[1]]$genes, fromType = "SYMBOL", toType = c("ENTREZID", :
“6.38% of input gene IDs are fail to map...”


,SYMBOL,ENTREZID,ENSEMBL
,<chr>,<chr>,<chr>
1,ABCB6,10058,ENSG00000115657
2,ARL5B,221079,ENSG00000165997
3,ATP12A,479,ENSG00000075673
4,ATP1B1,481,ENSG00000143153
5,ATP7B,540,ENSG00000123191
6,ATP8B2,57198,ENSG00000143515
8,CD164L2,388611,ENSG00000174950
9,CD226,10666,ENSG00000150637
10,CD3D,915,ENSG00000167286


In [10]:
enrichWP(ids$ENTREZID,
    organism = "Homo sapiens", pAdjustMethod = "BH",
    pvalueCutoff = 0.01,
    qvalueCutoff = 0.05
) %>% as.data.frame()


ID,Description,GeneRatio,BgRatio,RichFactor,FoldEnrichment,zScore,pvalue,p.adjust,qvalue,geneID,Count
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>


In [11]:
enrichKEGG(ids$ENTREZID,
        organism = "hsa", pAdjustMethod = "BH",
        pvalueCutoff = 0.01,
        qvalueCutoff = 0.05
) %>% as.data.frame()


,category,subcategory,ID,Description,GeneRatio,BgRatio,RichFactor,FoldEnrichment,zScore,pvalue,p.adjust,qvalue,geneID,Count
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
hsa04650,Organismal Systems,Immune system,hsa04650,Natural killer cell mediated cytotoxicity,4/17,134/9440,0.02985075,16.57594,7.712907,8.004587e-05,0.003602064,0.003286094,3824/4277/9436/7305,4


In [12]:
enrichGO(ids$ENTREZID, org.Hs.eg.db,
        ont = "BP", pAdjustMethod = "BH",
        pvalueCutoff = 0.01,
        qvalueCutoff = 0.05,
        readable = TRUE
) %>% as.data.frame()


,ID,Description,GeneRatio,BgRatio,RichFactor,FoldEnrichment,zScore,pvalue,p.adjust,qvalue,geneID,Count
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
GO:0007005,GO:0007005,mitochondrion organization,9/42,445/18805,0.02022472,9.055377,8.136323,4.785959e-07,0.0003383673,0.0002780894,DNAJC11/JTB/MTCH1/MTCH2/PLD6/RHOT1/SLC25A33/TRABD/USP30,9
GO:0008053,GO:0008053,mitochondrial fusion,4/42,35/18805,0.11428571,51.170068,14.055465,1.070098e-06,0.0003782797,0.0003108917,MTCH2/PLD6/TRABD/USP30,4
GO:0006839,GO:0006839,mitochondrial transport,6/42,183/18805,0.03278689,14.679938,8.798226,3.068008e-06,0.0007230272,0.0005942247,MTCH1/MTCH2/RHOT1/SLC25A16/SLC25A20/SLC25A33,6
GO:0001912,GO:0001912,positive regulation of leukocyte mediated cytotoxicity,4/42,68/18805,0.05882353,26.337535,9.903015,1.577930e-05,0.0016557148,0.0013607602,CD226/KLRC4/KLRD1/TYROBP,4
GO:0002223,GO:0002223,stimulatory C-type lectin receptor signaling pathway,3/42,23/18805,0.13043478,58.400621,13.031914,1.778467e-05,0.0016557148,0.0013607602,KLRC4/KLRD1/TYROBP,3
GO:1990840,GO:1990840,response to lectin,3/42,23/18805,0.13043478,58.400621,13.031914,1.778467e-05,0.0016557148,0.0013607602,KLRC4/KLRD1/TYROBP,3
GO:1990858,GO:1990858,cellular response to lectin,3/42,23/18805,0.13043478,58.400621,13.031914,1.778467e-05,0.0016557148,0.0013607602,KLRC4/KLRD1/TYROBP,3
GO:0031343,GO:0031343,positive regulation of cell killing,4/42,71/18805,0.05633803,25.224681,9.675438,1.873510e-05,0.0016557148,0.0013607602,CD226/KLRC4/KLRD1/TYROBP,4
GO:0045954,GO:0045954,positive regulation of natural killer cell mediated cytotoxicity,3/42,29/18805,0.10344828,46.317734,11.554849,3.635336e-05,0.0028557587,0.0023470242,CD226/KLRC4/KLRD1,3


In [13]:
?enrichGO


enrichGO            package:clusterProfiler            R Documentation

_G_O _E_n_r_i_c_h_m_e_n_t _A_n_a_l_y_s_i_s _o_f _a _g_e_n_e _s_e_t. _G_i_v_e_n _a _v_e_c_t_o_r _o_f _g_e_n_e_s, _t_h_i_s
_f_u_n_c_t_i_o_n _w_i_l_l _r_e_t_u_r_n _t_h_e _e_n_r_i_c_h_m_e_n_t _G_O _c_a_t_e_g_o_r_i_e_s _a_f_t_e_r _F_D_R _c_o_n_t_r_o_l.

_D_e_s_c_r_i_p_t_i_o_n:

     GO Enrichment Analysis of a gene set. Given a vector of genes,
     this function will return the enrichment GO categories after FDR
     control.

_U_s_a_g_e:

     enrichGO(
       gene,
       OrgDb,
       keyType = "ENTREZID",
       ont = "MF",
       pvalueCutoff = 0.05,
       pAdjustMethod = "BH",
       universe,
       qvalueCutoff = 0.2,
       minGSSize = 10,
       maxGSSize = 500,
       readable = FALSE,
       pool = FALSE
     )
     
_A_r_g_u_m_e_n_t_s:

    gene: a vector of entrez gene i

In [14]:
all_vertices <- igraph::V(merged_networks)$name
target_proteins <- rna_mod_proteins$gene_symbol
target_in_network <- target_proteins[target_proteins %in% all_vertices]
length(target_in_network)
setdiff(target_proteins, all_vertices)


[1] 56

[1] "NSUN1" "DNMT2"

In [15]:
fc <- igraph::cluster_fast_greedy(merged_networks)
module_list <- igraph::groups(fc)


In [16]:
names(module_list[[1]])


NULL

In [17]:
louvain_clusters <- igraph::cluster_louvain(merged_networks)


In [18]:
louvain_members <- membership(louvain_clusters)
modularity_score <- modularity(louvain_clusters)
cat("Louvain模块度：", modularity_score, "\n")


ERROR: Error in membership(louvain_clusters): could not find function "membership"


In [ ]:
module_sizes <- table(louvain_members)
module_sizes


louvain_members
   1    2    3    4    5    6    7    8    9   10   11   12   13   14   15   16 
1865 2948 4831 5302 1405 2261 2619  223   30   60    2    5    7    2    1    9 
  17   18   19   20   21   22   23   24   25   26   27   28   29   30 
   2    2    3    2    3    2    2    2    2    2    4    2    2    4 

In [ ]:
louvain_module_df <- data.frame(
  gene = names(louvain_members),
  module_id = as.integer(louvain_members),
  stringsAsFactors = FALSE
)


In [ ]:
subgraph_genes <- louvain_module_df$gene[louvain_module_df$module_id %in% c(2, 3, 4, 5, 6, 7)]
rna_subgraph <- induced_subgraph(merged_networks, subgraph_genes)
cat("RNA修饰子网络规模：", vcount(rna_subgraph), "节点，", ecount(rna_subgraph), "边\n")


RNA修饰子网络规模： 19366 节点， 984347 边


In [ ]:
subgraph_genes <- louvain_module_df$gene[louvain_module_df$module_id %in% c(2)]
rna_subgraph <- induced_subgraph(merged_networks, subgraph_genes)
fine_clusters <- cluster_fast_greedy(rna_subgraph)


In [ ]:
fine_clusters


IGRAPH clustering fast greedy, groups: 39, mod: 0.43
+ groups:
  $`1`
    [1] "ABHD8"           "ADAT1"           "ADPRH"           "AGGF1"          
    [5] "AGO1"            "AGO2"            "AGO3"            "ALG13"          
    [9] "ALKBH1"          "ALKBH2"          "ALKBH6"          "ALKBH7"         
   [13] "ALKBH8"          "ANKHD1"          "ANKHD1-EIF4EBP3" "ANKLE1"         
   [17] "ANKRD17"         "ANKRD33B"        "ANKRD37"         "ANXA2R"         
   [21] "ASZ1"            "ATXN2"           "B9D2"            "BICC1"          
   [25] "BTG3"            "BTG4"            "C14ORF79"        "C6ORF48"        
   [29] "C8ORF58"         "C9ORF57"         "C9ORF84"         "C9ORF85"        
   [33] "CACTIN-AS1"      "CCDC179"         "CCDC3"           "CCDC83"         
  + ... omitted several groups/vertices

In [ ]:
louvain_module_df %>%
    dplyr::filter(gene %in% rna_mod_proteins$gene_symbol) %>%
    arrange(module_id)


gene,module_id
<chr>,<int>
ADARB2,2
ALKBH1,2
ALKBH5,2
ALYREF,2
FTO,2
HNRNPA2B1,2
IGF2BP1,2
IGF2BP2,2
IGF2BP3,2


In [ ]:
sapply(louvain_clusters, function(x) length(x))


membership memberships  modularity       names      vcount   algorithm 
       2682        4004        3630        5008         268        4110

In [ ]:
igraph::membership(louvain_clusters)
igraph::modularity(louvain_clusters)


                  A1BG                   A1CF                    A2M 
                     1                      2                      1 
                 A2ML1                A3GALT2                 A4GALT 
                     1                      3                      4 
                 A4GNT                   AAAS                   AACS 
                     4                      4                      3 
                 AADAC                  AADAT                  AAED1 
                     5                      3                      6 
                 AAGAB                   AAK1                  AAMDC 
                     4                      2                      2 
                  AAMP                  AANAT                   AAR2 
                     6                      5                      3 
                  AARD                   AARS                  AARS1 
                     2                      3                      7 
                 AAR

[1] 0.3166346

In [ ]:
fc <- igraph::cluster_fast_greedy(merged_networks)
module_list <- igraph::groups(fc)
intersect(get_rna_mod_proteins()$gene_symbol, module_list[["6"]])


character(0)

In [ ]:
fc <- igraph::cluster_edge_betweenness(merged_networks)
module_list <- igraph::groups(fc)
intersect(get_rna_mod_proteins()$gene_symbol, module_list[["6"]])


In [ ]:
names(module_list)


[1] "1"  "2"  "3"  "4"  "5"  "6"  "7"  "8"  "9"  "10" "11" "12" "13" "14" "15"
[16] "16" "17" "18" "19" "20" "21" "22" "23" "24" "25" "26" "27" "28" "29" "30"
[31] "31" "32" "33" "34" "35" "36" "37" "38" "39" "40" "41" "42" "43" "44" "45"
[46] "46" "47" "48" "49" "50" "51" "52" "53" "54" "55" "56" "57" "58" "59" "60"
[61] "61" "62" "63" "64" "65" "66" "67" "68" "69" "70" "71" "72" "73" "74" "75"
[76] "76" "77" "78" "79"

In [ ]:
module_list[["5"]]


[1] "ART1"     "OR52N1"   "OR52N5"   "SEPTIN1"  "SEPTIN10" "SEPTIN11"
 [7] "SEPTIN12" "SEPTIN14" "SEPTIN2"  "SEPTIN3"  "SEPTIN4"  "SEPTIN5" 
[13] "SEPTIN6"  "SEPTIN7"  "SEPTIN8"  "SEPTIN9"  "TMEM250"

In [ ]:
merged_networks_i <- merge_ppi_networks(networkds, merge_method = "intersection", add_source_labels = TRUE)


Merging 3 networks using method: intersection



=== Merged Network Summary ===
Data sources: string, biogrid, intact
Merge method: intersection
Total nodes: 3802
Total edges: 6113
Network density: 0.0008
Average degree: 3.22

=== Source Contribution ===

biogrid  intact  string 
   6113    6113    6113 

Interactions from multiple sources: 6113 (100.0%)


In [ ]:
merged_networks_i


IGRAPH 80f364c UNW- 3802 6113 -- 
+ attr: name (v/c), degree (v/n), weight (e/n), sources (e/c)
+ edges from 80f364c (vertex names):
 [1] AAGAB   --AP2S1   AAK1    --AP2M1   AAK1    --AP2S1   AAR2    --EFTUD2 
 [5] AASDHPPT--USP22   AATF    --NGDN    ABCB7   --FECH    ABHD16A --RNF5   
 [9] ABI1    --ABL1    ABI1    --CYFIP1  ABI1    --NCKAP1  ABI1    --WASF1  
[13] ABI1    --WASF2   ABI2    --BRK1    ABI2    --CYFIP1  ABI2    --NCK2   
[17] ABI2    --NCKAP1  ABI2    --TRIM32  ABI2    --WASF1   ABL1    --CRK    
[21] ABL1    --CRKL    ABL1    --RIN1    ABL1    --YWHAB   ABL1    --YWHAE  
[25] ABL1    --YWHAG   ABL1    --YWHAH   ABL1    --YWHAZ   ABL2    --CRK    
[29] ABL2    --RIN1    ABTB1   --CUL3    ABTB1   --EEF1A2  ACAD9   --ECSIT  
+ ... omitted several edges

In [ ]:
networkds


$string
IGRAPH 37070ab UN-- 16185 236000 -- 
+ attr: name (v/c), source (e/c)
+ edges from 37070ab (vertex names):
 [1] ARF5 --ACAP1     ARF5 --COPA      ARF5 --RAB11FIP3 ARF5 --COPB2    
 [5] ARF5 --COPE      ARF5 --ARF3      ARF5 --ARFGAP1   ARF5 --RAB11FIP1
 [9] ARF5 --CYTH1     ARF5 --IQSEC1    ARF5 --RAB11A    ARF5 --ARFGAP2  
[13] ARF5 --COPB1     ARF5 --ARF4      ARF5 --ASAP1     ARF5 --RAB11FIP4
[17] ARF5 --GORAB     ARF5 --ACAP2     ARF5 --ARFGAP3   ARF5 --ARFIP1   
[21] ARF5 --ARF1      ARF5 --CDKN2A    ARF5 --GBF1      ARF5 --ARFIP2   
[25] ARF5 --AGAP1     ARF5 --ASAP2     ARF5 --COPZ1     M6PR --GGA3     
[29] M6PR --VTI1A     M6PR --TGOLN2    M6PR --GGA2      M6PR --GGA1     
+ ... omitted several edges

$biogrid
IGRAPH 21b965d UN-- 20169 959426 -- 
+ attr: name (v/c), source (e/c)
+ edges from 21b965d (vertex names):
 [1] MAP2K4--FLNC     MYPN  --ACTN2    ACVR1 --FNTA     GATA2 --PML     
 [5] RPA2  --STAT3    ARF1  --GGA3     ARF3  --ARFIP2   ARF3  --ARFIP1  
 [9] XRN1 

In [ ]:
visualize_ppi_network(networkds[["string"]], interactive = TRUE)


Warning message in visualize_ppi_network(networkds[["string"]], interactive = TRUE):
“visNetwork package not available, falling back to static plot”


ERROR: Error in cut.default(edge_weights, breaks = length(unique(edge_weights))): invalid number of intervals


In [ ]:
read.csv("data-raw/rna_modification_proteins.csv") %>%
    dplyr::select(gene_symbol, modification_type, functional_role) -> df

saveRDS(df, "inst/extdata/rnamod_proteins.rds")


In [ ]:
visualize_ppi_network(networkds[["string"]],
    layout_type = "fr",
    node_color = "degree",
    edge_color = "weight",
    node_size = "degree",
    show_labels = TRUE,
    highlight_nodes = NULL,
    title = "PPI Network",
    save_path = NULL,
    width = 10,
    height = 8,
    interactive = FALSE
)


ERROR: Error in cut.default(edge_weights, breaks = length(unique(edge_weights))): invalid number of intervals


In [ ]:
# test <- preprocessing_string(
#     species_code = 9606,
#     score_threshold = 0,
#     output_file = "inst/extdata/stringdb_9606_v12.rds",
#     version = "12",
#     verbose = TRUE
# )


In [ ]:
biogrid_file <- "data-raw/BIOGRID-ALL-4.4.248.tab3.txt"
biogrid_data <- data.table::fread(biogrid_file, sep = "\t", check.names = TRUE, quote = "")


In [ ]:
verbose <- TRUE


In [ ]:
if (verbose) {
  message(sprintf("Loaded %d interactions from BioGRID", nrow(biogrid_data)))
}


Loaded 2804534 interactions from BioGRID



In [ ]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [ ]:
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == 9606)



if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$gene1, processed_data$gene2)))))
}


Processed 1219381 unique interactions

Unique proteins: 20334



In [ ]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [ ]:
species_code <- 9606
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::mutate(species_code = species_code, est = factor(Experimental.System.Type, levels = c("physical", "genetic"))) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == species_code) %>%
  arrange(est) %>%
  distinct(gene1, gene2, .keep_all = TRUE) %>%
  dplyr::select(gene1, gene2, species_code, est)


In [ ]:
saveRDS(processed_data, "inst/extdata/biogrid_9606_v4.4.248.rds")


In [ ]:
readRDS("inst/extdata/intact_9606_v20250328.rds")


gene1,gene2,species_code,score
<chr>,<chr>,<chr>,<dbl>
MDM2,TP53,9606,1.00
TP53,MDM2,9606,1.00
CDK4,CCND1,9606,0.99
CCND1,CDK4,9606,0.99
CCNT1,CDK9,9606,0.98
EIF4EBP1,EIF4E,9606,0.98
BCL2L1,BAD,9606,0.98
CASP8,FADD,9606,0.98
HRAS,RAF1,9606,0.98


In [ ]:
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == 9606)



if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$gene1, processed_data$gene2)))))
}


Processed 1219381 unique interactions

Unique proteins: 20334



In [ ]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [ ]:
# Process the data to required format
# BioGRID format processing
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Score = Score
  ) %>%
  dplyr::mutate(
    # Convert gene IDs to character and clean
    geneA = as.character(.data$geneA),
    geneB = as.character(.data$geneB),

    # Ensure species codes are character
    species_codeA = as.character(.data$species_codeA),
    species_codeB = as.character(.data$species_codeB),

    # BioGRID doesn't have confidence scores, use 1.0 as default
    score = 1.0,

    # Filter for specified species
    species_code = .data$species_codeA
  ) %>%
  # Filter for specified species
  dplyr::filter(
    .data$species_codeA == species_code,
    .data$species_codeB == species_code,
    .data$species_codeA == .data$species_codeB
  ) %>%
  # Clean and finalize
  dplyr::mutate(
    geneA = toupper(.data$geneA),
    geneB = toupper(.data$geneB)
  ) %>%
  dplyr::select(geneA, geneB, score, species_code) %>%
  dplyr::distinct() %>%
  dplyr::filter(
    !is.na(.data$geneA),
    !is.na(.data$geneB),
    .data$geneA != .data$geneB,
    .data$geneA != "",
    .data$geneB != "",
    .data$geneA != "-",
    .data$geneB != "-"
  )

# Clean up temporary files
unlink(temp_zip)
unlink(extract_dir, recursive = TRUE)

if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$geneA, processed_data$geneB)))))
}

# Save if output file specified
if (!is.null(output_file)) {
  saveRDS(processed_data, file = output_file)
  if (verbose) {
    message(sprintf("Data saved to: %s", output_file))
  }
}

return(processed_data)


ERROR: [1m[33mError[39m in `dplyr::mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `geneA = as.character(.data$geneA)`.
[1mCaused by error in `.data$geneA`:[22m
[1m[22m[33m![39m Column `geneA` not found in `.data`.


In [ ]:
a <- list(
  string_version = "12"
)


In [ ]:
a[["string_version"]]


[1] "12"

In [ ]:

    
    if (verbose) {
      message(sprintf("Loaded %d interactions from BioGRID", nrow(biogrid_data)))
    }
    
    # Process the data to required format
    # BioGRID format processing
    processed_data <- biogrid_data %>%
      dplyr::select(
        # BioGRID column names
        geneA = .data$`Entrez Gene Interactor A`,
        geneB = .data$`Entrez Gene Interactor B`,
        species_codeA = .data$`Organism ID Interactor A`,
        species_codeB = .data$`Organism ID Interactor B`,
        experimental_system = .data$`Experimental System`,
        pubmed_id = .data$`Pubmed ID`
      ) %>%
      dplyr::mutate(
        # Convert gene IDs to character and clean
        geneA = as.character(.data$geneA),
        geneB = as.character(.data$geneB),
        
        # Ensure species codes are character
        species_codeA = as.character(.data$species_codeA),
        species_codeB = as.character(.data$species_codeB),
        
        # BioGRID doesn't have confidence scores, use 1.0 as default
        score = 1.0,
        
        # Filter for specified species
        species_code = .data$species_codeA
      ) %>%
      # Filter for specified species
      dplyr::filter(
        .data$species_codeA == species_code,
        .data$species_codeB == species_code,
        .data$species_codeA == .data$species_codeB
      ) %>%
      # Clean and finalize
      dplyr::mutate(
        geneA = toupper(.data$geneA),
        geneB = toupper(.data$geneB)
      ) %>%
      dplyr::select(geneA, geneB, score, species_code) %>%
      dplyr::distinct() %>%
      dplyr::filter(
        !is.na(.data$geneA),
        !is.na(.data$geneB),
        .data$geneA != .data$geneB,
        .data$geneA != "",
        .data$geneB != "",
        .data$geneA != "-",
        .data$geneB != "-"
      )
    
    # Clean up temporary files
    unlink(temp_zip)
    unlink(extract_dir, recursive = TRUE)
    
    if (verbose) {
      message(sprintf("Processed %d unique interactions", nrow(processed_data)))
      message(sprintf("Unique proteins: %d", length(unique(c(processed_data$geneA, processed_data$geneB)))))
    }
    
    # Save if output file specified
    if (!is.null(output_file)) {
      saveRDS(processed_data, file = output_file)
      if (verbose) {
        message(sprintf("Data saved to: %s", output_file))
      }
    }
    
    return(processed_data)
    
  }, error = function(e) {
    stop("Error processing BioGRID data: ", e$message)
  })
}


In [ ]:
library(dplyr)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
species_code <- 9606


In [ ]:
# Process the data to required format
# IntAct PSI-MITAB format processing
processed_data <- intact_data %>%
    dplyr::select(
        Alt..ID.s..interactor.A,
        Alt..ID.s..interactor.B,
        Alias.es..interactor.A,
        Alias.es..interactor.B,
        Taxid.interactor.A,
        Taxid.interactor.B,
        Confidence.value.s.
    ) %>%
    # Extract species information
    dplyr::mutate(
        taxidA = stringr::str_extract(Taxid.interactor.A, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:"),
        taxidB = stringr::str_extract(Taxid.interactor.B, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:")
    ) %>%
    # Filter by species
    dplyr::filter(
        !is.na(taxidA), !is.na(taxidB),
        taxidA == species_code,
        taxidB == species_code,
        taxidA == taxidB
    ) %>%
    # Extract gene names using multiple patterns
    dplyr::mutate(
        # Extract from gene name annotations
        geneA_name = stringr::str_extract(
            Alias.es..interactor.A,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),
        geneB_name = stringr::str_extract(
            Alias.es..interactor.B,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),

        # Extract from display names as fallback
        geneA_display = stringr::str_extract(
            Alias.es..interactor.A,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),
        geneB_display = stringr::str_extract(
            Alias.es..interactor.B,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),

        # Use gene name if available, otherwise display name
        gene1 = dplyr::coalesce(geneA_name, geneA_display),
        gene2 = dplyr::coalesce(geneB_name, geneB_display),

        # Extract interaction score
        score = stringr::str_extract(Confidence.value.s., "intact-miscore:([0-9\\.]+)") %>%
            stringr::str_remove("intact-miscore:") %>%
            as.numeric(),
        species_code = taxidA
    ) %>%
    # Clean and filter
    dplyr::select(gene1, gene2, score, species_code) %>%
    dplyr::filter(
        !is.na(gene1), !is.na(gene2), gene1 != "", gene2 != "", gene1 != gene2,
        !is.na(score),
    ) %>%
    dplyr::distinct() %>%
    dplyr::arrange(dplyr::desc(score)) %>%
    dplyr::select(gene1, gene2, species_code, score)


In [ ]:
saveRDS(processed_data, "inst/extdata/intact_9606_v20250328.rds")


In [ ]:
if (verbose) {
    message(sprintf("Loaded %d interactions from IntAct", nrow(intact_data)))
}

# Process the data to required format
# IntAct PSI-MITAB format processing
processed_data <- intact_data %>%
    dplyr::select(
        Alt..ID.s..interactor.A,
        Alt..ID.s..interactor.B,
        Alias.es..interactor.A,
        Alias.es..interactor.B,
        Taxid.interactor.A,
        Taxid.interactor.B,
        Confidence.value.s.
    ) %>%
    # Extract species information
    dplyr::mutate(
        taxidA = stringr::str_extract(Taxid.interactor.A, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:"),
        taxidB = stringr::str_extract(Taxid.interactor.B, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:")
    ) %>%
    # Filter by species
    dplyr::filter(
        !is.na(taxidA), !is.na(taxidB),
        taxidA %in% species_filter,
        taxidB %in% species_filter,
        taxidA == taxidB
    ) %>%
    # Extract gene names using multiple patterns
    dplyr::mutate(
        # Extract from gene name annotations
        geneA_name = stringr::str_extract(
            Alias.es..interactor.A,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),
        geneB_name = stringr::str_extract(
            Alias.es..interactor.B,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),

        # Extract from display names as fallback
        geneA_display = stringr::str_extract(
            Alias.es..interactor.A,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),
        geneB_display = stringr::str_extract(
            Alias.es..interactor.B,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),

        # Use gene name if available, otherwise display name
        geneA = dplyr::coalesce(geneA_name, geneA_display),
        geneB = dplyr::coalesce(geneB_name, geneB_display),

        # Extract interaction score
        score = stringr::str_extract(Confidence.value.s., "intact-miscore:([0-9\\.]+)") %>%
            stringr::str_remove("intact-miscore:") %>%
            as.numeric(),
        taxid = taxidA
    ) %>%
    # Clean and filter
    dplyr::select(geneA, geneB, score, taxid) %>%
    dplyr::filter(
        !is.na(geneA), !is.na(geneB), geneA != "", geneB != "",
        !is.na(score), score >= min_score
    ) %>%
    dplyr::distinct() %>%
    dplyr::arrange(dplyr::desc(score))
dplyr::mutate(
    # Extract UniProt IDs from complex identifiers
    geneA = stringr::str_extract(.data$geneA, "uniprotkb:([A-Za-z0-9-]+)") %>%
        stringr::str_remove("uniprotkb:"),
    geneB = stringr::str_extract(.data$geneB, "uniprotkb:([A-Za-z0-9-]+)") %>%
        stringr::str_remove("uniprotkb:"),

    # Extract species codes
    species_codeA = stringr::str_extract(.data$taxidA, "taxid:([0-9]+)") %>%
        stringr::str_remove("taxid:"),
    species_codeB = stringr::str_extract(.data$taxidB, "taxid:([0-9]+)") %>%
        stringr::str_remove("taxid:"),

    # Extract IntAct confidence score
    score = stringr::str_extract(.data$score_raw, "intact-miscore:([0-9\\.]+)") %>%
        stringr::str_remove("intact-miscore:") %>%
        as.numeric()
) %>%
    # Filter for specified species
    dplyr::filter(
        .data$species_codeA == species_code,
        .data$species_codeB == species_code,
        .data$species_codeA == .data$species_codeB
    ) %>%
    # Filter by score threshold
    dplyr::filter(
        !is.na(.data$score),
        .data$score >= score_threshold
    ) %>%
    # Clean and finalize
    dplyr::mutate(
        species_code = .data$species_codeA,
        geneA = toupper(.data$geneA),
        geneB = toupper(.data$geneB)
    ) %>%
    dplyr::select(geneA, geneB, score, species_code) %>%
    dplyr::distinct() %>%
    dplyr::filter(
        !is.na(.data$geneA),
        !is.na(.data$geneB),
        .data$geneA != .data$geneB,
        .data$geneA != "",
        .data$geneB != ""
    )


In [ ]:
test <- preprocessing_intact(
    species_code = "9606",
    score_threshold = 0.4,
    output_file = NULL,
    version = "2025-03-28",
    verbose = TRUE
)




To the file: /tmp/RtmpPOUGqD/file6fc22d54fd70.txt



Warning message in utils::download.file(intact_url, temp_file, quiet = !verbose, :
“downloaded length 21767737 != reported length 7890868954”
Warning message in utils::download.file(intact_url, temp_file, quiet = !verbose, :
“URL 'https://ftp.ebi.ac.uk/pub/databases/intact/2025-03-28/psimitab/intact.txt': Timeout of 60 seconds was reached”


ERROR: Error in utils::download.file(intact_url, temp_file, quiet = !verbose, : download from 'https://ftp.ebi.ac.uk/pub/databases/intact/2025-03-28/psimitab/intact.txt' failed


In [ ]:
download_ppi_data("STRING",
    output_dir = "data-raw/",
    processing_params = list(
        species_filter = c("9606"),
        min_score = 0.0
    ),
    overwrite = FALSE,
    verbose = TRUE
)


Warning message in download.file(config$url, paste0(temp_file, ".gz"), quiet = !verbose):
“downloaded length 0 != reported length 153”
Warning message in download.file(config$url, paste0(temp_file, ".gz"), quiet = !verbose):
“cannot open URL 'https://stringdb-static.org/download/protein.links.v12/9606.protein.links.v12.txt.gz': HTTP status was '404 Not Found'”
Warning message in value[[3L]](cond):
“Download failed for STRING: cannot open URL 'https://stringdb-static.org/download/protein.links.v12/9606.protein.links.v12.txt.gz'”
Warning message in value[[3L]](cond):
“Falling back to local file processing”


ERROR: Error in process_local_ppi_data(database_type, output_dir, processing_params, : Local STRING data file not found at: data-raw/string_interactions.txt


In [ ]:
df <- data.table::fread("data-raw/core.psimitab", sep = "\t", check.names = TRUE, quote = "", header = FALSE)


In [ ]:
df <- data.table::fread("data-raw/BIOGRID-ALL-4.4.248.tab3.txt", sep = "\t", check.names = TRUE, quote = "", header = TRUE)
df %>%
    dplyr::select(
        Official.Symbol.Interactor.A, Official.Symbol.Interactor.B,
        Organism.ID.Interactor.A,
        Organism.ID.Interactor.B, Organism.Name.Interactor.A, Organism.Name.Interactor.B, Throughput, Score, Modification,
        Experimental.System,
        Experimental.System.Type,
        Author,
        Publication.Source,
        Organism.ID.Interactor.A,
        Organism.ID.Interactor.B,
        Throughput,
        Score, Qualifications, Tags
    ) %>%
    dplyr::filter(Organism.ID.Interactor.A == Organism.ID.Interactor.B, Organism.ID.Interactor.A %in% c(9606, 10090), Organism.ID.Interactor.B %in% c(9606, 10090)) %>%
    arrange(Score) -> sadf
saveRDS(sadf, "inst/extdata/biogrid_processed.rds")


In [ ]:
df %>%
    pull(Experimental.System) %>%
    unique()


[1] "Two-hybrid"                    "Affinity Capture-MS"          
 [3] "Affinity Capture-Western"      "Dosage Rescue"                
 [5] "Reconstituted Complex"         "Synthetic Growth Defect"      
 [7] "Synthetic Lethality"           "Synthetic Rescue"             
 [9] "Affinity Capture-Luminescence" "Biochemical Activity"         
[11] "Co-crystal Structure"          "Far Western"                  
[13] "FRET"                          "Protein-peptide"              
[15] "Co-localization"               "Affinity Capture-RNA"         
[17] "Protein-RNA"                   "PCA"                          
[19] "Co-purification"               "Co-fractionation"             
[21] "Dosage Lethality"              "Phenotypic Enhancement"       
[23] "Phenotypic Suppression"        "Dosage Growth Defect"         
[25] "Negative Genetic"              "Positive Genetic"             
[27] "Synthetic Haploinsufficiency"  "Proximity Label-MS"           
[29] "Cross-Linking-MS (XL-MS)"      "Surface Display"

In [ ]:
df %>%
    dplyr::select(Throughput, Experimental.System, Experimental.System.Type, Score, Source.Database) %>%
    distinct()


Throughput,Experimental.System,Experimental.System.Type,Score,Source.Database
<chr>,<chr>,<chr>,<chr>,<chr>
Low Throughput,Two-hybrid,physical,-,BIOGRID
High Throughput,Two-hybrid,physical,-,BIOGRID
High Throughput,Affinity Capture-MS,physical,-,BIOGRID
Low Throughput,Affinity Capture-MS,physical,-,BIOGRID
Low Throughput,Affinity Capture-Western,physical,-,BIOGRID
Low Throughput,Dosage Rescue,genetic,-,BIOGRID
Low Throughput,Reconstituted Complex,physical,-,BIOGRID
High Throughput|Low Throughput,Synthetic Growth Defect,genetic,-,BIOGRID
Low Throughput,Synthetic Lethality,genetic,-,BIOGRID


In [ ]:
df %>% dplyr::select(
    Official.Symbol.Interactor.A, Official.Symbol.Interactor.B,
    Organism.ID.Interactor.A,
    Organism.ID.Interactor.B, Organism.Name.Interactor.A, Organism.Name.Interactor.B, Throughput, Score, Modification,
    Experimental.System,
    Experimental.System.Type,
    Author,
    Publication.Source,
    Organism.ID.Interactor.A,
    Organism.ID.Interactor.B,
    Throughput,
    Score, Qualifications, Tags
)


Official.Symbol.Interactor.A,Official.Symbol.Interactor.B,Organism.ID.Interactor.A,Organism.ID.Interactor.B,Organism.Name.Interactor.A,Organism.Name.Interactor.B,Throughput,Score,Modification,Experimental.System,Experimental.System.Type,Author,Publication.Source,Qualifications,Tags
<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
MAP2K4,FLNC,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Marti A (1997),PUBMED:9006895,-,-
MYPN,ACTN2,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Bang ML (2001),PUBMED:11309420,-,-
ACVR1,FNTA,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Wang T (1996),PUBMED:8599089,-,-
GATA2,PML,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Tsuzuki S (2000),PUBMED:10938104,-,-
RPA2,STAT3,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kim J (2000),PUBMED:10875894,-,-
ARF1,GGA3,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Dell'Angelica EC (2000),PUBMED:10747089,-,-
ARF3,ARFIP2,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kanoh H (1997),PUBMED:9038142,-,-
ARF3,ARFIP1,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kanoh H (1997),PUBMED:9038142,-,-
XRN1,ALDOA,9606,9606,Homo sapiens,Homo sapiens,High Throughput,-,-,Two-hybrid,physical,Lehner B (2004),PUBMED:15231747,-,-


In [ ]:
# Build PPI network using STRING database
net <- build_ppi_network(
    proteins = NULL, # Use full network
    data_sources = c("STRING", "IntAct"),
    min_confidence = 0.7,
    species = "Homo sapiens",
    string_version = "12"
)



# Summary statistics
cat("Network Summary:\n")
cat("Nodes:", length(igraph::V(net)), "\n")
cat("Edges:", length(igraph::E(net)), "\n")
cat("Average degree:", mean(degree(net)), "\n")


ℹ Loading MPAGE




Warning message:
“Objects listed as exports, but not present in namespace:
• add_custom_proteins
• save_rna_mod_proteins”



STRING DB Network Summary:
Nodes: 16201 
Edges: 473860 
Average degree: 58.49762 


Loading IntAct network from: /workspaces/MPAGE/inst/extdata/intact_processed.rds



IntAct DB Network Summary:
Nodes: 5784 
Edges: 21915 
Average degree: 7.577801 
Network Summary:
Nodes: 16544 
Edges: 248963 
Average degree: 30.09707 



Loading BioGRID data from: /tmp/RtmpNNgF4Q/BIOGRID-ALL-4.4.248.tab3.txt

Warning message in FUN(file, exdir = tmpdir):
“error -1 in extracting from zip file”


ERROR: Error in value[[3L]](cond): Error reading BioGRID file: File is empty: /tmp/RtmpNNgF4Q/BIOGRID-ALL-4.4.248.tab3.txt


In [ ]:
get_string_ppi <- function(species_code, string_version, output_file) {
    tryCatch(
        {
            string_db <- STRINGdb::STRINGdb$new(
                species = species_code,
                version = string_version,
                score_threshold = 0
            )

            # Get full network instead of protein-limited network
            message("Downloading full STRING network for species ", species_code)
            stringdb_proteins <- string_db$get_proteins()
            # Get all interactions for the species above threshold
            all_interactions <- string_db$get_interactions(
                stringdb_proteins$protein_external_id
            )

            if (nrow(all_interactions) == 0) {
                return(NULL)
            }

            # Map interactions to gene symbols
            all_interactions$gene1 <- stringdb_proteins$preferred_name[match(all_interactions$from, stringdb_proteins$protein_external_id)]
            all_interactions$gene2 <- stringdb_proteins$preferred_name[match(all_interactions$to, stringdb_proteins$protein_external_id)]

            # Filter valid interactions
            all_interactions <- all_interactions[!is.na(all_interactions$gene1) & !is.na(all_interactions$gene2), ]

            saveRDS(all_interactions[, c("gene1", "gene2", "combined_score")], output_file)
            return(all_interactions[, c("gene1", "gene2", "combined_score")])
        },
        error = function(e) {
            warning("Error accessing STRING database: ", e$message)
            return(NULL)
        }
    )
}


In [ ]:
interactions <- get_string_ppi(9606, "12", "test.rds")


In [ ]:
species_code <- 9606
string_version <- "12"
sprintf("stringdb_%s_%s.rds", as.character(species_code), string_version)


[1] "stringdb_9606_12.rds"

In [ ]:
interactions[, c("gene1", "gene2", "combined_score")]


,gene1,gene2,combined_score
,<chr>,<chr>,<int>
1,ARF5,RALGPS2,173
2,ARF5,FHDC1,154
3,ARF5,ATP6V1E1,151
4,ARF5,CYTH2,471
5,ARF5,PSD3,201
6,ARF5,TTC9C,180
7,ARF5,SLC2A4,181
8,ARF5,GGA1,594
9,ARF5,RCC1L,154


In [ ]:
library(igraph)



Attaching package: ‘igraph’


The following object is masked from ‘package:testthat’:

    compare


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union




In [ ]:
g <- make_graph("Zachary")

## We put everything into a big 'try' block, in case
## igraph was compiled without GLPK support

## The calculation only takes a couple of seconds
oc <- cluster_optimal(g)

## Double check the result
print(modularity(oc))
print(modularity(g, membership(oc)))

## Compare to the greedy optimizer
fc <- cluster_fast_greedy(g)
print(modularity(fc))


[1] 0.4197896
[1] 0.4197896
[1] 0.3806706


In [21]:
# GSE26378

# 加载包
library(GEOquery) # 下载GEO数据
library(limma) # 差异表达分析
library(pheatmap) # 热图可视化
library(ggplot2) # 基础可视化
library(dplyr) # 数据处理


Setting options('download.file.method.GEOquery'='auto')

Setting options('GEOquery.inmemory.gpl'=FALSE)


Attaching package: ‘limma’


The following object is masked from ‘package:BiocGenerics’:

    plotMA



Attaching package: ‘dplyr’


The following object is masked from ‘package:AnnotationDbi’:

    select


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following object is masked from ‘package:Biobase’:

    combine


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, setequal, union


The following object is masked from ‘package:generics’:

    explain


The following object is masked from ‘package:MPAGE’:

    as_data_frame


The following object is masked from ‘package:testthat’:

    matches


The following objects are masked from ‘package:stats’:



In [23]:
# 下载GEO数据（根据网络情况可能需要几分钟）
gse_id <- "GSE26378" # 替换为你的目标数据集ID
gse <- getGEO(gse_id, destdir = ".", getGPL = TRUE) # getGPL=FALSE加快速度

# 提取表达矩阵和表型信息（通常取第一个平台的数据）
expr_set <- gse[[1]] # 表达矩阵（包含探针ID和表达值）
pdata <- pData(expr_set) # 样本表型信息（如处理组/对照组）

# 查看数据基本信息
cat("样本数量：", ncol(expr_set), "\n")
cat("探针数量：", nrow(expr_set), "\n")
head(pdata[, c("title", "source_name_ch1")]) # 查看样本分组信息


Found 1 file(s)

GSE26378_series_matrix.txt.gz

Using locally cached version: ./GSE26378_series_matrix.txt.gz



样本数量： 103 
探针数量： 54675 


,title,source_name_ch1
,<chr>,<chr>
GSM647527,Septic Shock_biological rep1,Children ≤ 10 years old
GSM647528,Septic Shock_biological rep2,Children ≤ 10 years old
GSM647529,Septic Shock_biological rep3,Children ≤ 10 years old
GSM647530,Septic Shock_biological rep4,Children ≤ 10 years old
GSM647531,Septic Shock_biological rep5,Children ≤ 10 years old
GSM647532,Septic Shock_biological rep6,Children ≤ 10 years old


In [27]:
exprs(expr_set)


,GSM647527,GSM647528,GSM647529,GSM647530,GSM647531,GSM647532,GSM647533,GSM647534,GSM647535,GSM647536,⋯,GSM647620,GSM647621,GSM647622,GSM647623,GSM647624,GSM647625,GSM647626,GSM647627,GSM647628,GSM647629
1007_s_at,0.478,0.497,0.968,0.831,0.642,0.808,1.152,0.995,0.684,1.140,⋯,0.701,0.769,1.185,0.959,0.904,0.925,0.960,0.612,0.510,0.750
1053_at,1.005,1.028,1.199,1.306,1.343,1.714,0.763,0.846,0.686,0.848,⋯,1.204,1.368,0.816,1.189,1.153,0.912,0.782,1.017,0.957,1.098
117_at,2.992,3.027,1.637,3.192,0.773,0.921,0.416,1.513,2.530,0.889,⋯,2.691,0.719,0.492,0.714,0.314,0.707,0.583,2.074,1.110,0.612
121_at,0.770,0.771,0.825,0.698,0.785,0.954,1.017,1.033,1.039,0.767,⋯,1.021,0.986,1.178,1.132,0.902,1.139,1.364,0.919,0.897,0.932
1255_g_at,1.190,1.178,1.075,0.843,0.860,0.871,0.931,1.001,0.950,0.963,⋯,1.072,0.949,0.892,0.940,0.856,0.974,1.019,1.141,1.031,1.070
1294_at,1.080,1.051,1.533,1.132,1.143,1.237,1.172,1.264,0.603,1.278,⋯,0.621,0.971,0.780,1.013,0.888,1.212,0.652,0.789,0.507,0.912
1316_at,0.689,0.787,0.691,0.773,0.744,0.946,1.151,1.233,0.872,0.796,⋯,0.893,1.075,1.311,1.227,1.094,1.243,1.022,0.911,0.649,1.084
1320_at,1.128,1.189,0.984,0.951,1.082,1.070,0.999,0.888,0.923,0.843,⋯,1.041,0.969,1.043,1.027,0.991,1.042,0.977,1.048,0.849,1.012
1405_i_at,0.161,0.154,1.173,0.872,0.349,2.299,1.127,1.052,0.465,1.763,⋯,0.134,0.545,0.828,1.479,1.187,2.180,0.444,0.151,0.180,0.150
1431_at,0.927,1.039,0.951,0.926,0.931,1.205,0.915,0.988,0.901,0.847,⋯,0.921,0.931,0.982,0.924,0.916,0.895,0.900,1.122,1.042,0.951
